In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p> Implement Causal (masked) Self-Attention for a given set of matrices.
  Given the query matrix <code>Q</code> of size <code>M×d</code>, key matrix <code>K</code> of size <code>M×d</code>, and value matrix
  <code>V</code> of size <code>M×d</code>, your program should compute the output matrix using the formula:
  $$\text{Attention}_{\text{causal}}(Q, K, V) = \text{softmax}\Bigl(\text{masked}\Bigl( \frac{QK^T}{\sqrt{d}} \Bigr)\Bigr)V$$
</p>


<p>
  where <code>mask</code> is a causal mask that sets all positions corresponding to keys <strong>after</strong> the current query to $-\infty$.
  $$$$
  i.e., for query <code>i</code> and key <code>j</code>:
  $$
  \text{masked}(a_{ij}) =
  \begin{cases}
  a_{ij}, & j \le i \\
  -\infty, & j > i
  \end{cases}
  $$
  The softmax function is applied row-wise. <code>Q</code>, <code>K</code>, <code>V</code>, and <code>output</code> are all of data type <code>float32</code>;
  <code>M</code>, and <code>d</code> are of data type <code>int32</code>.
</p>

<svg width="310" height="310" viewBox="0 0 310 310" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;" font-family="monospace" font-size="10">
  <rect width="310" height="310" rx="8" fill="#222"/>

  <!-- Axis labels -->
  <text x="175" y="16" text-anchor="middle" fill="#999" font-size="10">key position &#x2192;</text>
  <text x="14" y="170" fill="#999" font-size="10" transform="rotate(-90,14,170)">query &#x2192;</text>

  <!-- Column headers -->
  <text x="62"  y="34" text-anchor="middle" fill="#777">0</text>
  <text x="100" y="34" text-anchor="middle" fill="#777">1</text>
  <text x="138" y="34" text-anchor="middle" fill="#777">2</text>
  <text x="176" y="34" text-anchor="middle" fill="#777">3</text>
  <text x="214" y="34" text-anchor="middle" fill="#777">4</text>
  <text x="252" y="34" text-anchor="middle" fill="#777">5</text>

  <!-- Row headers -->
  <text x="32" y="58"  text-anchor="middle" fill="#777">0</text>
  <text x="32" y="96"  text-anchor="middle" fill="#777">1</text>
  <text x="32" y="134" text-anchor="middle" fill="#777">2</text>
  <text x="32" y="172" text-anchor="middle" fill="#777">3</text>
  <text x="32" y="210" text-anchor="middle" fill="#777">4</text>
  <text x="32" y="248" text-anchor="middle" fill="#777">5</text>

  <!-- 6x6 grid: lower triangle = attend (blue), upper = masked (dark red) -->
  <!-- Row 0 -->
  <rect x="44"  y="42" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="82"  y="42" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="120" y="42" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="158" y="42" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="196" y="42" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="234" y="42" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <!-- Row 1 -->
  <rect x="44"  y="80" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="82"  y="80" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="120" y="80" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="158" y="80" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="196" y="80" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="234" y="80" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <!-- Row 2 -->
  <rect x="44"  y="118" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="82"  y="118" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="120" y="118" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="158" y="118" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="196" y="118" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="234" y="118" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <!-- Row 3 -->
  <rect x="44"  y="156" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="82"  y="156" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="120" y="156" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="158" y="156" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="196" y="156" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <rect x="234" y="156" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <!-- Row 4 -->
  <rect x="44"  y="194" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="82"  y="194" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="120" y="194" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="158" y="194" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="196" y="194" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="234" y="194" width="36" height="28" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <!-- Row 5 -->
  <rect x="44"  y="232" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="82"  y="232" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="120" y="232" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="158" y="232" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="196" y="232" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <rect x="234" y="232" width="36" height="28" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>

  <!-- Checkmarks in attend cells -->
  <!-- Row 0 -->
  <text x="62"  y="60" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <!-- Row 1 -->
  <text x="62"  y="98" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="100" y="98" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <!-- Row 2 -->
  <text x="62"  y="136" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="100" y="136" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="138" y="136" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <!-- Row 3 -->
  <text x="62"  y="174" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="100" y="174" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="138" y="174" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="176" y="174" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <!-- Row 4 -->
  <text x="62"  y="212" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="100" y="212" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="138" y="212" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="176" y="212" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="214" y="212" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <!-- Row 5 -->
  <text x="62"  y="250" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="100" y="250" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="138" y="250" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="176" y="250" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="214" y="250" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>
  <text x="252" y="250" text-anchor="middle" fill="#66cc66" font-size="12">&#x2713;</text>

  <!-- Text in masked cells: -∞ -->
  <!-- Row 0 -->
  <text x="100" y="60" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="138" y="60" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="176" y="60" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="214" y="60" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="252" y="60" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <!-- Row 1 -->
  <text x="138" y="98" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="176" y="98" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="214" y="98" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="252" y="98" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <!-- Row 2 -->
  <text x="176" y="136" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="214" y="136" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="252" y="136" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <!-- Row 3 -->
  <text x="214" y="174" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <text x="252" y="174" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>
  <!-- Row 4 -->
  <text x="252" y="212" text-anchor="middle" fill="#cc6666" font-size="9">-&#x221e;</text>

  <!-- Diagonal boundary highlight -->
  <line x1="82"  y1="42" x2="82"  y2="70" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="82"  y1="70" x2="118" y2="70" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="118" y1="70" x2="118" y2="108" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="118" y1="108" x2="156" y2="108" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="156" y1="108" x2="156" y2="146" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="156" y1="146" x2="194" y2="146" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="194" y1="146" x2="194" y2="184" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="194" y1="184" x2="232" y2="184" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="232" y1="184" x2="232" y2="222" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>
  <line x1="232" y1="222" x2="270" y2="222" stroke="#ffffff" stroke-width="1.5" stroke-opacity="0.3"/>

  <!-- Legend -->
  <rect x="44" y="275" width="14" height="14" rx="2" fill="#2a4a7a" stroke="#3a5a8a"/>
  <text x="64" y="287" fill="#999" font-size="10">attend</text>
  <rect x="160" y="275" width="14" height="14" rx="2" fill="#3d1e1e" stroke="#4d2e2e"/>
  <text x="180" y="287" fill="#999" font-size="10">masked</text>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The
    <code>solve</code> function signature must remain unchanged
  </li>
  <li>The final result must be stored in the output matrix
    <code>output</code>
  </li>
</ul>
<h2>Example 1:</h2>
<p>
<strong>Input:</strong><br>
<code>Q</code> (2×4):
$$
\begin{bmatrix}
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0
\end{bmatrix}
$$
<code>K</code> (2×4):
$$
\begin{bmatrix}
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0
\end{bmatrix}
$$
<code>V</code> (2×4):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0
\end{bmatrix}
$$
</p>

<p>
<strong>Output:</strong><br>
<code>output</code> (2×4):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
3.4898374 & 4.4898374 & 5.4898374 & 6.4898374
\end{bmatrix}
$$
</p>


<h2>Example 2:</h2>
<p>
<strong>Input:</strong><br>
<code>Q</code> (2×2):
$$
\begin{bmatrix}
0.0 & 0.0 \\
1.0 & 1.0
\end{bmatrix}
$$
<code>K</code> (2×2):
$$
\begin{bmatrix}
1.0 & 0.0 \\
0.0 & 1.0
\end{bmatrix}
$$
<code>V</code> (2×2):
$$
\begin{bmatrix}
3.0 & 4.0 \\
5.0 & 6.0
\end{bmatrix}
$$
</p>

<p>
<strong>Output:</strong><br>
<code>output</code> (2×2):
$$
\begin{bmatrix}
3.0 & 4.0 \\
5.0 & 6.0
\end{bmatrix}
$$
</p>


<h2>Constraints</h2>
<ul>
  <li>Matrix <code>Q</code>, <code>K</code>, and <code>V</code> are all of size <code>M×d</code></li>
  <li>1 &le; <code>M</code> &le; 10000</li>
  <li>1 &le; <code>d</code> &le; 128</li>
  <li>All elements in <code>Q</code>, <code>K</code>, and <code>V</code> are sampled from<code>[-100.0, 100.0]</code></li>
  <li>Data type for all matrices is <code>float32</code></li>

  <li>Performance is measured with <code>M</code> = 5,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// Q, K, V, output are device pointers
extern "C" void solve(const float* Q, const float* K, const float* V, float* output, int M, int d) {

}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# Q, K, V, output are tensors on the GPU
@cute.jit
def solve(
    Q: cute.Tensor,
    K: cute.Tensor,
    V: cute.Tensor,
    output: cute.Tensor,
    M: cute.Int32,
    d: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# Q, K, V are tensors on the GPU
@jax.jit
def solve(Q: jax.Array, K: jax.Array, V: jax.Array, M: int, d: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# Q, K, V, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    Q: UnsafePointer[Float32, MutExternalOrigin],
    K: UnsafePointer[Float32, MutExternalOrigin],
    V: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    d: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# Q, K, V, output are tensors on the GPU
def solve(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, output: torch.Tensor, M: int, d: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# Q, K, V, output are tensors on the GPU
def solve(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, output: torch.Tensor, M: int, d: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/hard/53_casual_attention/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
